In [1]:
'''Librerias'''
import pandas as pd
import numpy as np
import unicodedata
from math import radians, sin, cos, sqrt, atan2
from tqdm import tqdm
import pyarrow.parquet as pq
import re
import unicodedata

In [2]:
'''Funciones'''
# Carga csv, xlsx, parquet
def cargar_archivo_filtrado_por_batches(file_path: str, batch_size: int = 1000_000) -> pd.DataFrame:
    """
    Carga un archivo (CSV, XLSX o Parquet) en batches y filtra filas sin coordenadas.

    Parámetros:
    - file_path: ruta al archivo
    - batch_size: tamaño del batch

    Retorna:
    - DataFrame con lat/lon válidas de todo el archivo
    """
    # Paso 1: determinar el tipo de archivo y contar filas para saber cuántos batches habrá
    if file_path.endswith('.csv'):
        total_filas = sum(1 for _ in open(file_path, encoding='utf-8')) - 1  # -1 para excluir encabezado
    elif file_path.endswith('.xlsx'):
        total_filas = pd.read_excel(file_path, engine='openpyxl').shape[0]
    elif file_path.endswith('.parquet'):
        # Usamos pyarrow para obtener el número total de filas en un archivo Parquet
        parquet_file = pq.ParquetFile(file_path)
        total_filas = parquet_file.metadata.num_rows
    else:
        raise ValueError(f"Formato de archivo no soportado: {file_path}")

    total_batches_estimado = (total_filas + batch_size - 1) // batch_size

    dfs = []
    total_rows = 0
    batch_num = 0

    print(f"Iniciando carga del archivo: {file_path}")
    print(f"Total de filas detectadas: {total_filas}")
    print(f"Se estiman {total_batches_estimado} batches (de tamaño {batch_size})\n")

    # Usando tqdm para la barra de progreso
    with tqdm(total=total_batches_estimado, desc="Procesando batches", unit="batch") as pbar:
        if file_path.endswith('.csv'):
            # Leer el archivo CSV en batches
            for chunk in pd.read_csv(file_path, chunksize=batch_size, low_memory=False):
                batch_num += 1
                rows_before = len(chunk)
                total_rows += rows_before
                dfs.append(chunk)
                pbar.update(1)

        elif file_path.endswith('.xlsx'):
            # Leer el archivo XLSX completo en memoria
            df_xlsx = pd.read_excel(file_path, engine='openpyxl')
            # Dividir el DataFrame en batches
            for start in range(0, len(df_xlsx), batch_size):
                batch_num += 1
                chunk = df_xlsx.iloc[start:start+batch_size]
                rows_before = len(chunk)
                total_rows += rows_before
                dfs.append(chunk)
                pbar.update(1)

        elif file_path.endswith('.parquet'):
            # Leer el archivo Parquet en batches (usando pyarrow)
            parquet_file = pq.ParquetFile(file_path)
            num_row_groups = parquet_file.num_row_groups
            for i in range(num_row_groups):
                # Leemos un grupo de filas de Parquet
                row_group = parquet_file.read_row_group(i)
                df_parquet = row_group.to_pandas()

                # Dividimos en chunks dentro del row group
                for start in range(0, len(df_parquet), batch_size):
                    batch_num += 1
                    chunk = df_parquet.iloc[start:start+batch_size]
                    rows_before = len(chunk)
                    total_rows += rows_before
                    dfs.append(chunk)
                    pbar.update(1)

    df_completo = pd.concat(dfs, ignore_index=True)
    print(f"\nCarga completa. Total de batches procesados: {batch_num}")
    print(f"Filas totales cargadas: {total_rows}")

    return df_completo

def normalize_text(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize('NFKD', s)
    return "".join(c for c in s if not unicodedata.combining(c)).lower().strip()

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(float, [lat1, lon1, lat2, lon2])
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlambda = radians(lon2 - lon1)
    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlambda/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1-a))

def normalize_name(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()

    # quitar acentos
    s = "".join(
        ch for ch in unicodedata.normalize("NFKD", s)
        if not unicodedata.combining(ch)
    )

    # quitar puntuación rara / dejar letras, números, & y espacios
    s = re.sub(r"[^a-z0-9&\s]", " ", s)

    # colapsar espacios
    s = re.sub(r"\s+", " ", s).strip()
    return s

def remove_chains_from_df(
    df: pd.DataFrame,
    name_col: str = "name",
    keep_pharmacies: bool = True
):
    out = df.copy()

    # Normalizar
    out["name_norm"] = out[name_col].map(normalize_name)

    # Flags
    out["is_chain"] = out["name_norm"].apply(lambda x: bool(chain_re.search(x)))
    out["is_pharmacy"] = out["name_norm"].apply(lambda x: bool(pharm_re.search(x)))

    # Si NO quieres borrar farmacias, las protegemos
    if keep_pharmacies:
        out["remove_chain"] = out["is_chain"] & (~out["is_pharmacy"])
    else:
        out["remove_chain"] = out["is_chain"]

    # Stats
    total = len(out)
    removed = int(out["remove_chain"].sum())
    kept = total - removed
    unique_chain_names = out.loc[out["remove_chain"], "name_norm"].nunique()

    summary = {
        "total_rows": total,
        "removed_rows": removed,
        "kept_rows": kept,
        "unique_chain_names_removed": unique_chain_names,
    }

    # Top cadenas por volumen (para que veas qué está pegando más)
    top_chains = (
        out.loc[out["remove_chain"]]
           .groupby("name_norm")
           .size()
           .sort_values(ascending=False)
           .head(30)
           .reset_index(name="rows")
    )

    # DF final sin cadenas
    df_clean = out.loc[~out["remove_chain"]].copy()

    return df_clean, out, summary, top_chains



In [3]:
#%% CONFIGURACIÓN
path_datamach = r"C:\Users\MFML1\Desktop\Kin Analytics\VTM\Prospección\Mexico\Golfo\Datamatch\20261012_nocturno_golfo_fuzzy_8_9\final_output.csv"
path_clientes = r"C:\Users\MFML1\Desktop\Kin Analytics\VTM\Prospección\Mexico\Golfo\Clientes\MV2.csv"

#%% CARGA DE DATOS
# Cargar Dataplor en batches
df_datamatch = cargar_archivo_filtrado_por_batches(
    path_datamach,
    batch_size=1000_000)

df_clientes = cargar_archivo_filtrado_por_batches(
    path_clientes,
    batch_size=1000_000)

Iniciando carga del archivo: C:\Users\MFML1\Desktop\Kin Analytics\VTM\Prospección\Mexico\Golfo\Datamatch\20261012_nocturno_golfo_fuzzy_8_9\final_output.csv
Total de filas detectadas: 34387
Se estiman 1 batches (de tamaño 1000000)



Procesando batches: 100%|██████████| 1/1 [00:00<00:00, 13.76batch/s]



Carga completa. Total de batches procesados: 1
Filas totales cargadas: 6860
Iniciando carga del archivo: C:\Users\MFML1\Desktop\Kin Analytics\VTM\Prospección\Mexico\Golfo\Clientes\MV2.csv
Total de filas detectadas: 1767
Se estiman 1 batches (de tamaño 1000000)



Procesando batches: 100%|██████████| 1/1 [00:00<00:00, 176.17batch/s]


Carga completa. Total de batches procesados: 1
Filas totales cargadas: 1767


In [4]:
#%% Fix encoding BOM
df_clientes.columns = df_clientes.columns.str.replace("ï»¿", "", regex=False)

#%% Crear columnas consistentes de lat/lon
df_clientes["lat_cli"] = pd.to_numeric(df_clientes["pos_latitude"], errors="coerce")
df_clientes["lon_cli"] = pd.to_numeric(df_clientes["pos_longitude"], errors="coerce")

df_datamatch["lat_dp"] = pd.to_numeric(df_datamatch["latitude.c"], errors="coerce")
df_datamatch["lon_dp"] = pd.to_numeric(df_datamatch["longitude.c"], errors="coerce")

#%% Normalización de nombres

df_datamatch["name_norm"] = df_datamatch["name.c"].apply(normalize_text)
df_clientes["pos_name_norm"] = df_clientes["pos_name"].apply(normalize_text)

#%% Filtrar no matches
CHOSEN_THRESHOLD = 0.71  # <── CAMBIA ESTE VALOR

df_datamatch.loc[df_datamatch['total_score'] < CHOSEN_THRESHOLD, 'total_score'] = float('nan')
#df_datamatch.loc[df_datamatch['distance_score'] > 0.90, 'total_score'] = df_datamatch['distance_score']-0.35

df_no_match = df_datamatch[df_datamatch['total_score'].isna()]

In [5]:
df_no_match.shape

(6531, 32)

In [6]:
#%% Merge por nombre normalizado
df_nm = df_no_match.merge(
    df_clientes[['pos_id', "pos_name", "pos_name_norm", "lat_cli", "lon_cli"]],
    left_on="name_norm",
    right_on="pos_name_norm",
    how="inner"
)

#%% Calcular distancia correcta
df_nm["distance_m"] = df_nm.apply(
    lambda r: haversine(r["lat_dp"], r["lon_dp"], r["lat_cli"], r["lon_cli"]),
    axis=1
)

#%% Filtrar por radio
df_nm_filtrado = df_nm[df_nm["distance_m"] < 100].copy()

#%% Mantener match único por dataplor_id
df_nm_filtrado = df_nm_filtrado.sort_values(["id.c", "distance_m"]) \
                               .drop_duplicates("id.c", keep="first")

df_casos_nombre = df_nm_filtrado[['id.c','name.c','pos_id', 'pos_name', 'distance_m']].copy()
print(df_casos_nombre.head())

#%% APLICAR MATCH POR NOMBRE A df_datamatch#%% Tomar matches confirmados por nombre
df_casos_nombre = df_nm_filtrado[['id.c','pos_id']].copy()
df_casos_nombre = df_casos_nombre.rename(columns={'pos_id': 'pos_id_nuevo'})

# Crear diccionario dataplor_id → pos_id
map_nuevos = dict(zip(df_casos_nombre['id.c'], df_casos_nombre['pos_id_nuevo']))

#%% Actualizar pos_id SOLO para estos dataplor_id
df_datamatch['id.o'] = df_datamatch.apply(
    lambda r: map_nuevos.get(r['id.c'], r['id.o']),
    axis=1
)

#%% Actualizar match_no_match_final SOLO para estos dataplor_id
df_datamatch['match_no_match'] = np.where(df_datamatch["total_score"] > 0, "match", "no_match")

df_datamatch['match_no_match'] = df_datamatch.apply(
    lambda r: 'match' if r['id.c'] in map_nuevos else r['match_no_match'],
    axis=1
)

print("\nActualización completada:")
print(df_datamatch['match_no_match'].value_counts())
print(df_datamatch['match_no_match'].value_counts(normalize=True))

Empty DataFrame
Columns: [id.c, name.c, pos_id, pos_name, distance_m]
Index: []

Actualización completada:
match_no_match
no_match    6531
match        329
Name: count, dtype: int64
match_no_match
no_match    0.952041
match       0.047959
Name: proportion, dtype: float64


In [7]:
#Nos quedamos con columnas necesarias
columns_to_keep = ['id.c', 'name.c', 'address.c','latitude.c', 'longitude.c',
                   'id.o', 'name.o', 'latitude.o', 'longitude.o',
                   'distance_score', 'name_score','address_score', 'phonetic_score',
                   'total_score', 'match_no_match']

df_datamatch = df_datamatch[columns_to_keep].copy()

#Se renombra las columnas
df_datamatch = df_datamatch.rename(columns={'id.c':'dataplor_id','name.c':'dataplor_name',
                                           'address.c':'dataplor_address','latitude.c':'dataplor_latitude',
                                           'longitude.c':'dataplor_longitude','id.o':'pos_id',
                                           'name.o':'pos_name','address.o':'pos_address',
                                           'latitude.o':'pos_latitude','longitude.o':'pos_longitude'})

#Se guarda la base completa
out_path_total = r"C:\Users\MFML1\Desktop\Kin Analytics\VTM\Prospección\Mexico\Golfo\Datamatch\Datamatch_Final_Golfo_20261013.csv"
df_datamatch.to_csv(out_path_total, index=False)

'''Se guarda Lat/Lon'''

columns_to_keep = ['dataplor_id', 'dataplor_name', 'dataplor_address', 'dataplor_latitude',
       'dataplor_longitude', 'match_no_match']

df_datamatch_latlon = df_datamatch[columns_to_keep].copy()

#Se guarda la base completa
#out_path_total = r"C:\Users\MFML1\Desktop\Kin Analytics\VTM\Prospección\Mexico\Arca\Jul2026\Datamatch\20260723_Datamatch_Arca_Guadalajara\Datamatch_Final_Arca_Guadalajara_LatLon_20260723.csv"
#df_datamatch_latlon.to_csv(out_path_total, index=False)